# Case Study 3: Financial Time Series — Stock Prices

## RNN vs LSTM vs GRU: The Efficient Market Hypothesis Stress Test

---

### Objective
Predict next-day closing price of AAPL stock using three recurrent architectures:
- **Vanilla RNN** — simple recurrence, prone to vanishing gradients
- **LSTM** — gated memory cells (forget, input, output gates)
- **GRU** — simplified gating (reset, update gates)

### KEY PEDAGOGICAL GOAL
> **This notebook is intentionally designed to show that LSTMs DO NOT beat naive baselines
> for stock price prediction.** Students often expect deep learning to "solve" stock prediction.
> This case study systematically demonstrates why that expectation is wrong.

### The Efficient Market Hypothesis (EMH)
The EMH states that asset prices fully reflect all available information. In its semi-strong form,
no amount of historical price analysis (technical analysis) should yield consistent excess returns.
If true, the best predictor of tomorrow's price is simply **today's price** — the naive baseline.

### What You Will Learn
1. How to preprocess financial time series with proper feature engineering
2. Building and comparing RNN, LSTM, GRU for stock price forecasting
3. **Why the "1-day shift" illusion makes LSTM look good but isn't**
4. **Why naive baselines are essential for honest evaluation**
5. The difference between predicting prices vs. predicting returns
6. Cross-ticker generalization testing

### Dataset
- **Primary**: AAPL (Apple Inc.) daily prices, ~2765 trading days (2015–2025)
- **Generalization**: GOOGL, MSFT, TSLA
- Source: Yahoo Finance via `yfinance`

---
## 2. Environment Setup

In [ ]:
import random
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {DEVICE}')

# ---- Plot style ----
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_style('whitegrid')

COLORS = {'RNN': '#e74c3c', 'LSTM': '#2ecc71', 'GRU': '#3498db'}
TICKER_COLORS = {'AAPL': '#333333', 'GOOGL': '#4285F4', 'MSFT': '#00A4EF', 'TSLA': '#CC0000'}

---
## 3. Exploratory Data Analysis

We load all four tickers and perform comprehensive analysis before focusing on AAPL for modeling.

In [ ]:
def load_yfinance_csv(filepath):
    """Load a yfinance CSV with multi-row header and flatten columns.

    yfinance CSVs have format:
        Row 0: Price, Close, High, Low, Open, Volume
        Row 1: Ticker, AAPL, AAPL, AAPL, AAPL, AAPL
        Row 2: Date, (empty cells)
    """
    df = pd.read_csv(filepath, header=[0, 1], index_col=0)
    # Flatten multi-level columns: take first level (Price type)
    df.columns = [col[0] for col in df.columns]
    df.index.name = 'Date'
    df.index = pd.to_datetime(df.index)
    # Convert to numeric
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.sort_index()
    df = df.dropna()
    return df

# Load all four tickers
DATA_DIR = 'Time_Series_Forecasting/stock_prices'
tickers = ['AAPL', 'GOOGL', 'MSFT', 'TSLA']
stock_data = {}

for ticker in tickers:
    filepath = f'{DATA_DIR}/{ticker}_stock.csv'
    stock_data[ticker] = load_yfinance_csv(filepath)
    print(f'{ticker}: {stock_data[ticker].shape[0]} rows, '
          f'{stock_data[ticker].index[0].date()} to {stock_data[ticker].index[-1].date()}')

# Primary dataset
aapl = stock_data['AAPL'].copy()
print(f'\nAAPL columns: {aapl.columns.tolist()}')
aapl.head(10)

In [ ]:
# Summary statistics for AAPL
print('AAPL Summary Statistics:')
print('=' * 60)
aapl.describe().round(2)

In [ ]:
# --- Plot 1: Price history overlay for all 4 tickers ---
fig, ax = plt.subplots(figsize=(16, 6))

for ticker in tickers:
    df_t = stock_data[ticker]
    ax.plot(df_t.index, df_t['Close'], label=ticker,
            color=TICKER_COLORS[ticker], linewidth=1.2, alpha=0.9)

ax.set_title('Closing Price History — All Tickers (2015–2025)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Price (USD)')
ax.legend(fontsize=11, loc='upper left')
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

print('Note the vastly different price scales — this is why we normalize before modeling.')

In [ ]:
# --- Plot 2: Volume analysis ---
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, ticker in zip(axes.flat, tickers):
    df_t = stock_data[ticker]
    ax.bar(df_t.index, df_t['Volume'], color=TICKER_COLORS[ticker], alpha=0.5, width=2)
    ax.set_title(f'{ticker} — Daily Trading Volume', fontsize=12)
    ax.set_ylabel('Volume')
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

    # Add moving average
    vol_ma = df_t['Volume'].rolling(60).mean()
    ax.plot(df_t.index, vol_ma, color='black', linewidth=1.5, label='60-day MA')
    ax.legend(fontsize=9)

plt.suptitle('Trading Volume Analysis', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 3: Daily returns distribution ---
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, ticker in zip(axes.flat, tickers):
    df_t = stock_data[ticker]
    returns = df_t['Close'].pct_change().dropna()

    ax.hist(returns, bins=100, color=TICKER_COLORS[ticker], alpha=0.7,
            edgecolor='white', density=True)

    # Overlay normal distribution
    mu, sigma = returns.mean(), returns.std()
    x = np.linspace(returns.min(), returns.max(), 200)
    ax.plot(x, stats.norm.pdf(x, mu, sigma), 'k--', linewidth=1.5, label='Normal fit')

    ax.set_title(f'{ticker} Daily Returns (mean={mu:.4f}, std={sigma:.4f})', fontsize=11)
    ax.set_xlabel('Daily Return')
    ax.legend()

    # Kurtosis
    kurt = stats.kurtosis(returns)
    skew = stats.skew(returns)
    ax.text(0.98, 0.95, f'Skew: {skew:.2f}\nKurtosis: {kurt:.2f}',
            transform=ax.transAxes, ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.suptitle('Daily Returns Distribution — Heavy Tails vs Normal', fontsize=14)
plt.tight_layout()
plt.show()

print('Key observation: Returns have HEAVY TAILS (high kurtosis) — more extreme moves')
print('than a normal distribution would predict. This makes forecasting very difficult.')

In [ ]:
# --- Plot 4: Rolling volatility ---
fig, ax = plt.subplots(figsize=(16, 6))

for ticker in tickers:
    df_t = stock_data[ticker]
    returns = df_t['Close'].pct_change().dropna()
    rolling_vol = returns.rolling(30).std() * np.sqrt(252)  # Annualized
    ax.plot(df_t.index[1:], rolling_vol, label=ticker,
            color=TICKER_COLORS[ticker], linewidth=1, alpha=0.8)

ax.set_title('30-Day Rolling Volatility (Annualized)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Annualized Volatility')
ax.legend(fontsize=11)
ax.axhline(y=0.2, color='gray', linestyle=':', alpha=0.5, label='20% benchmark')
plt.tight_layout()
plt.show()

print('Volatility clusters: high-vol periods tend to follow high-vol periods.')
print('TSLA consistently shows highest volatility — hardest to predict.')

In [ ]:
# --- Plot 5: Correlation between tickers (rolling 60-day) ---
# Build a DataFrame of close prices for correlation
close_prices = pd.DataFrame({t: stock_data[t]['Close'] for t in tickers})
close_prices = close_prices.dropna()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Static correlation
corr = close_prices.corr()
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdYlGn', center=0.5,
            square=True, ax=axes[0], vmin=0, vmax=1)
axes[0].set_title('Price-Level Correlation (Spurious!)', fontsize=12)

# Return correlation (meaningful)
returns_df = close_prices.pct_change().dropna()
corr_ret = returns_df.corr()
sns.heatmap(corr_ret, annot=True, fmt='.3f', cmap='RdYlGn', center=0.5,
            square=True, ax=axes[1], vmin=0, vmax=1)
axes[1].set_title('Daily Returns Correlation (Meaningful)', fontsize=12)

plt.suptitle('Correlation Analysis — Prices vs Returns', fontsize=14)
plt.tight_layout()
plt.show()

print('WARNING: Price-level correlations are SPURIOUS (non-stationary trending series).')
print('Returns correlations are meaningful: AAPL, GOOGL, MSFT move together (~0.5-0.7).')
print('TSLA is less correlated with the others.')

In [ ]:
# --- Plot 6: Log-returns analysis ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Log returns for AAPL
log_returns = np.log(aapl['Close'] / aapl['Close'].shift(1)).dropna()

axes[0].plot(aapl.index[1:], log_returns, color='#333333', linewidth=0.5, alpha=0.7)
axes[0].set_title('AAPL Log Returns (Daily)', fontsize=12)
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Log Return')
axes[0].axhline(0, color='red', linewidth=0.8, alpha=0.5)

# QQ plot
stats.probplot(log_returns, dist="norm", plot=axes[1])
axes[1].set_title('AAPL Log Returns — QQ Plot vs Normal', fontsize=12)
axes[1].get_lines()[0].set_color('#3498db')
axes[1].get_lines()[0].set_markersize(3)

plt.tight_layout()
plt.show()

print(f'Log returns stats:')
print(f'  Mean:     {log_returns.mean():.6f}')
print(f'  Std:      {log_returns.std():.6f}')
print(f'  Skewness: {stats.skew(log_returns):.4f}')
print(f'  Kurtosis: {stats.kurtosis(log_returns):.4f}')
print(f'\nQQ plot shows heavy tails — extreme moves are more common than normal distribution predicts.')

In [ ]:
# --- Plot 7: Autocorrelation of returns ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

returns_aapl = aapl['Close'].pct_change().dropna().values

# ACF of returns
n_lags = 40
acf_vals = [np.corrcoef(returns_aapl[:-lag], returns_aapl[lag:])[0, 1] for lag in range(1, n_lags+1)]
axes[0].bar(range(1, n_lags+1), acf_vals, color='#3498db', alpha=0.7, edgecolor='white')
axes[0].axhline(1.96/np.sqrt(len(returns_aapl)), color='red', linestyle='--', alpha=0.5)
axes[0].axhline(-1.96/np.sqrt(len(returns_aapl)), color='red', linestyle='--', alpha=0.5)
axes[0].set_title('ACF of Daily Returns', fontsize=12)
axes[0].set_xlabel('Lag (days)')
axes[0].set_ylabel('Autocorrelation')

# ACF of absolute returns (volatility clustering)
abs_returns = np.abs(returns_aapl)
acf_abs = [np.corrcoef(abs_returns[:-lag], abs_returns[lag:])[0, 1] for lag in range(1, n_lags+1)]
axes[1].bar(range(1, n_lags+1), acf_abs, color='#e74c3c', alpha=0.7, edgecolor='white')
axes[1].axhline(1.96/np.sqrt(len(abs_returns)), color='red', linestyle='--', alpha=0.5)
axes[1].axhline(-1.96/np.sqrt(len(abs_returns)), color='red', linestyle='--', alpha=0.5)
axes[1].set_title('ACF of |Returns| (Volatility Clustering)', fontsize=12)
axes[1].set_xlabel('Lag (days)')
axes[1].set_ylabel('Autocorrelation')

plt.tight_layout()
plt.show()

print('Critical insight:')
print('- Returns show almost NO autocorrelation → past returns don\'t predict future returns')
print('- |Returns| show STRONG autocorrelation → volatility clusters (GARCH effect)')
print('- This is WHY stock price prediction with LSTMs is fundamentally limited')

---
## 4. Data Preprocessing

We focus on AAPL for the primary modeling pipeline. Feature engineering adds derived columns
that capture momentum, volatility, and trend information.

In [ ]:
# Feature engineering for AAPL
df_model = aapl[['Close', 'High', 'Low', 'Open', 'Volume']].copy()

# Derived features
df_model['Returns'] = df_model['Close'].pct_change()
df_model['Log_Returns'] = np.log(df_model['Close'] / df_model['Close'].shift(1))
df_model['Rolling_Mean_5'] = df_model['Close'].rolling(5).mean()
df_model['Rolling_Mean_20'] = df_model['Close'].rolling(20).mean()
df_model['Rolling_Std_5'] = df_model['Close'].rolling(5).std()

# Drop NaN rows from rolling calculations
df_model = df_model.dropna().copy()
print(f'Shape after feature engineering: {df_model.shape}')
print(f'Features: {df_model.columns.tolist()}')
df_model.head()

In [ ]:
# Chronological split: 70% train, 15% val, 15% test
n = len(df_model)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df_model.iloc[:train_end].copy()
val_df = df_model.iloc[train_end:val_end].copy()
test_df = df_model.iloc[val_end:].copy()

print(f'Train: {len(train_df)} rows ({train_df.index[0].date()} to {train_df.index[-1].date()})')
print(f'Val:   {len(val_df)} rows ({val_df.index[0].date()} to {val_df.index[-1].date()})')
print(f'Test:  {len(test_df)} rows ({test_df.index[0].date()} to {test_df.index[-1].date()})')

# Visualize the split
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(train_df.index, train_df['Close'], color='#2ecc71', label=f'Train ({len(train_df)})', linewidth=1)
ax.plot(val_df.index, val_df['Close'], color='#f39c12', label=f'Validation ({len(val_df)})', linewidth=1)
ax.plot(test_df.index, test_df['Close'], color='#e74c3c', label=f'Test ({len(test_df)})', linewidth=1)
ax.set_title('AAPL — Chronological Train/Validation/Test Split', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Close Price (USD)')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Scale features using MinMaxScaler (fit on TRAIN only)
feature_cols = df_model.columns.tolist()
N_FEATURES = len(feature_cols)
TARGET_IDX = feature_cols.index('Close')  # We predict Close price

scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_df.values)
val_scaled = scaler.transform(val_df.values)
test_scaled = scaler.transform(test_df.values)

print(f'Number of features: {N_FEATURES}')
print(f'Target column: Close (index {TARGET_IDX})')
print(f'Scaled train range: [{train_scaled.min():.4f}, {train_scaled.max():.4f}]')

In [ ]:
# Sliding window dataset
SEQ_LENGTH = 30  # 30 trading days lookback (~6 weeks)

def create_sequences(data, seq_length, target_idx=0):
    """Create sliding window sequences.

    Args:
        data: scaled array (n_samples, n_features)
        seq_length: lookback window
        target_idx: column index for target

    Returns:
        X: (n_windows, seq_length, n_features)
        y: (n_windows, 1)
    """
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length, target_idx])
    return torch.FloatTensor(np.array(X)), torch.FloatTensor(np.array(y)).unsqueeze(1)


class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# Create sequences
X_train, y_train = create_sequences(train_scaled, SEQ_LENGTH, TARGET_IDX)
X_val, y_val = create_sequences(val_scaled, SEQ_LENGTH, TARGET_IDX)
X_test, y_test = create_sequences(test_scaled, SEQ_LENGTH, TARGET_IDX)

print(f'Training:   X={X_train.shape}, y={y_train.shape}')
print(f'Validation: X={X_val.shape}, y={y_val.shape}')
print(f'Test:       X={X_test.shape}, y={y_test.shape}')

In [ ]:
# DataLoaders
BATCH_SIZE = 32

train_loader = DataLoader(StockDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(StockDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(StockDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')
print(f'Test batches:  {len(test_loader)}')

# Visualize a sample window
fig, ax = plt.subplots(figsize=(12, 4))
sample_idx = 100
window = X_train[sample_idx, :, TARGET_IDX].numpy()
target = y_train[sample_idx, 0].numpy()

ax.plot(range(SEQ_LENGTH), window, 'o-', color='#3498db', label='Input window (Close)', markersize=4)
ax.plot(SEQ_LENGTH, target, 's', color='#e74c3c', markersize=10, label='Target (next day Close)', zorder=5)
ax.axvline(SEQ_LENGTH - 0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Time step (trading days)')
ax.set_ylabel('Scaled Close Price')
ax.set_title(f'Sliding Window Example (window={SEQ_LENGTH} days)', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Model Architecture

We use the same `SequenceForecaster` pattern from previous notebooks. Additionally, we implement
two critical baselines:
1. **NaiveBaseline**: Predict yesterday's price as tomorrow's price
2. **MovingAverageBaseline**: Predict the 5-day moving average

In [ ]:
class SequenceForecaster(nn.Module):
    """Unified RNN/LSTM/GRU model for sequence forecasting."""

    SUPPORTED_TYPES = {'RNN': nn.RNN, 'LSTM': nn.LSTM, 'GRU': nn.GRU}

    def __init__(self, model_type, input_size, hidden_size, output_size=1,
                 num_layers=1, dropout=0.0, bidirectional=False):
        super().__init__()
        assert model_type in self.SUPPORTED_TYPES, f'Unknown model_type: {model_type}'

        self.model_type = model_type
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional

        rnn_cls = self.SUPPORTED_TYPES[model_type]
        self.rnn = rnn_cls(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional
        )

        fc_input = hidden_size * (2 if bidirectional else 1)
        self.fc = nn.Linear(fc_input, output_size)

    def forward(self, x):
        rnn_out, _ = self.rnn(x)       # (batch, seq_len, hidden)
        last_hidden = rnn_out[:, -1, :] # Take last timestep
        return self.fc(last_hidden)     # (batch, output_size)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

In [ ]:
class NaiveBaseline:
    """Predict yesterday's close price as today's prediction.

    This is the gold standard baseline for stock prediction.
    If your model can't beat this, it has learned nothing useful.
    """
    def predict(self, X, target_idx=0):
        # X: (n_samples, seq_length, n_features)
        # Return the last Close value in each window
        if isinstance(X, torch.Tensor):
            return X[:, -1, target_idx].unsqueeze(1).numpy()
        return X[:, -1, target_idx].reshape(-1, 1)


class MovingAverageBaseline:
    """Predict using 5-day moving average of close prices."""
    def __init__(self, window=5):
        self.window = window

    def predict(self, X, target_idx=0):
        if isinstance(X, torch.Tensor):
            X_np = X.numpy()
        else:
            X_np = X
        # Average last `window` Close values
        return X_np[:, -self.window:, target_idx].mean(axis=1).reshape(-1, 1)


naive = NaiveBaseline()
ma5 = MovingAverageBaseline(window=5)

print('Baselines initialized:')
print('  NaiveBaseline: y_hat = yesterday\'s close')
print('  MovingAverageBaseline: y_hat = mean(last 5 days close)')

In [ ]:
# Parameter count comparison
print('Parameter Counts (hidden=64, layers=2, input_features=10):')
print('=' * 50)

param_data = []
for mt in ['RNN', 'LSTM', 'GRU']:
    model = SequenceForecaster(mt, input_size=N_FEATURES, hidden_size=64,
                                num_layers=2, dropout=0.1)
    p = model.count_parameters()
    param_data.append({'Model': mt, 'Parameters': p})
    print(f'  {mt:5s}: {p:>8,} parameters')

param_data.append({'Model': 'Naive', 'Parameters': 0})
param_data.append({'Model': 'MA-5', 'Parameters': 0})

fig, ax = plt.subplots(figsize=(8, 4))
pdf = pd.DataFrame(param_data)
bars = ax.bar(pdf['Model'], pdf['Parameters'],
              color=[COLORS.get(m, '#95a5a6') for m in pdf['Model']],
              edgecolor='white', width=0.5)
for bar, val in zip(bars, pdf['Parameters']):
    if val > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                f'{val:,}', ha='center', fontweight='bold', fontsize=10)
ax.set_title('Parameter Count Comparison', fontsize=13)
ax.set_ylabel('Trainable Parameters')
plt.tight_layout()
plt.show()

---
## 6. Training

Standard training loop with MSE loss, Adam optimizer, early stopping, and gradient clipping.

In [ ]:
def train_model(model, train_loader, X_val, y_val, epochs, lr,
                device=DEVICE, patience=10, clip_grad=1.0, verbose=True):
    """Train with early stopping and gradient clipping."""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=5, factor=0.5, verbose=False
    )

    history = {'train_loss': [], 'val_loss': []}
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0

    X_val_d = X_val.to(device)
    y_val_d = y_val.to(device)

    start = time.time()

    for epoch in range(epochs):
        model.train()
        batch_losses = []
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_b), y_b)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            optimizer.step()
            batch_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_val_d), y_val_d).item()

        avg_train = np.mean(batch_losses)
        history['train_loss'].append(avg_train)
        history['val_loss'].append(val_loss)
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                if verbose:
                    print(f'  Early stopping at epoch {epoch+1}')
                break

        if verbose and (epoch + 1) % 20 == 0:
            print(f'  Epoch {epoch+1:3d}/{epochs} | Train: {avg_train:.6f} | Val: {val_loss:.6f}')

    elapsed = time.time() - start
    if best_state:
        model.load_state_dict(best_state)
        model = model.to(device)

    return history, elapsed

In [ ]:
def inverse_scale_target(scaled_values, scaler, target_idx, n_features):
    """Inverse transform only the target column from MinMaxScaler."""
    dummy = np.zeros((len(scaled_values), n_features))
    dummy[:, target_idx] = scaled_values.flatten()
    inv = scaler.inverse_transform(dummy)
    return inv[:, target_idx]


def evaluate_model(model, X_test, y_test, scaler, target_idx, n_features, device=DEVICE):
    """Evaluate model and return metrics + predictions in original scale."""
    model.eval()
    with torch.no_grad():
        preds_scaled = model(X_test.to(device)).cpu().numpy()

    preds = inverse_scale_target(preds_scaled, scaler, target_idx, n_features)
    actuals = inverse_scale_target(y_test.numpy(), scaler, target_idx, n_features)

    rmse = np.sqrt(mean_squared_error(actuals, preds))
    mae = mean_absolute_error(actuals, preds)
    mape = np.mean(np.abs((actuals - preds) / actuals)) * 100
    r2 = r2_score(actuals, preds)

    return {'RMSE': rmse, 'MAE': mae, 'MAPE': f'{mape:.2f}%', 'R2': r2}, preds, actuals


def evaluate_baseline(baseline, X_test, y_test, scaler, target_idx, n_features):
    """Evaluate a baseline model."""
    preds_scaled = baseline.predict(X_test, target_idx=target_idx)
    preds = inverse_scale_target(preds_scaled, scaler, target_idx, n_features)
    actuals = inverse_scale_target(y_test.numpy(), scaler, target_idx, n_features)

    rmse = np.sqrt(mean_squared_error(actuals, preds))
    mae = mean_absolute_error(actuals, preds)
    mape = np.mean(np.abs((actuals - preds) / actuals)) * 100
    r2 = r2_score(actuals, preds)

    return {'RMSE': rmse, 'MAE': mae, 'MAPE': f'{mape:.2f}%', 'R2': r2}, preds, actuals

In [ ]:
# Quick training run with default hyperparameters to verify pipeline
print('Quick verification run (LSTM, 10 epochs)...')
test_model = SequenceForecaster('LSTM', input_size=N_FEATURES, hidden_size=64, num_layers=1)
test_history, _ = train_model(test_model, train_loader, X_val, y_val,
                               epochs=10, lr=0.001, verbose=True)
print('Pipeline verified!')

---
## 7. Hyperparameter Tuning

12 configurations x 3 model types = 36 trials with reduced training budget.

In [ ]:
# Hyperparameter search space: 12 configurations
SEARCH_CONFIGS = [
    {'hidden_size': 32,  'num_layers': 1, 'lr': 0.01,   'dropout': 0.0,  'seq_length': 30},
    {'hidden_size': 32,  'num_layers': 2, 'lr': 0.005,  'dropout': 0.1,  'seq_length': 30},
    {'hidden_size': 64,  'num_layers': 1, 'lr': 0.005,  'dropout': 0.0,  'seq_length': 30},
    {'hidden_size': 64,  'num_layers': 2, 'lr': 0.001,  'dropout': 0.1,  'seq_length': 30},
    {'hidden_size': 128, 'num_layers': 1, 'lr': 0.001,  'dropout': 0.0,  'seq_length': 30},
    {'hidden_size': 128, 'num_layers': 2, 'lr': 0.001,  'dropout': 0.2,  'seq_length': 30},
    {'hidden_size': 64,  'num_layers': 1, 'lr': 0.005,  'dropout': 0.0,  'seq_length': 15},
    {'hidden_size': 64,  'num_layers': 2, 'lr': 0.001,  'dropout': 0.1,  'seq_length': 15},
    {'hidden_size': 128, 'num_layers': 1, 'lr': 0.001,  'dropout': 0.0,  'seq_length': 60},
    {'hidden_size': 128, 'num_layers': 2, 'lr': 0.0005, 'dropout': 0.2,  'seq_length': 60},
    {'hidden_size': 256, 'num_layers': 1, 'lr': 0.0005, 'dropout': 0.0,  'seq_length': 30},
    {'hidden_size': 256, 'num_layers': 2, 'lr': 0.0005, 'dropout': 0.2,  'seq_length': 30},
]

TUNING_EPOCHS = 50

print(f'Search space: {len(SEARCH_CONFIGS)} configs x 3 model types = {len(SEARCH_CONFIGS)*3} total trials')
print(f'Tuning epochs per trial: {TUNING_EPOCHS}')

In [ ]:
tuning_results = []

for model_type in ['RNN', 'LSTM', 'GRU']:
    print(f'\n{"="*60}')
    print(f'Tuning {model_type}')
    print(f'{"="*60}')

    for i, config in enumerate(SEARCH_CONFIGS):
        seq_len = config['seq_length']

        # Rebuild sequences for this seq_length
        X_tr_c, y_tr_c = create_sequences(train_scaled, seq_len, TARGET_IDX)
        X_va_c, y_va_c = create_sequences(val_scaled, seq_len, TARGET_IDX)

        loader_c = DataLoader(StockDataset(X_tr_c, y_tr_c), batch_size=BATCH_SIZE, shuffle=True)

        model = SequenceForecaster(
            model_type=model_type,
            input_size=N_FEATURES,
            hidden_size=config['hidden_size'],
            output_size=1,
            num_layers=config['num_layers'],
            dropout=config['dropout']
        )

        history, t = train_model(model, loader_c, X_va_c, y_va_c,
                                  epochs=TUNING_EPOCHS, lr=config['lr'], verbose=False)

        best_val = min(history['val_loss'])
        tuning_results.append({
            'model_type': model_type, 'config_id': i,
            **config,
            'best_val_loss': best_val,
            'train_time': t,
            'epochs_run': len(history['val_loss']),
            'params': model.count_parameters()
        })

        print(f'  Config {i+1:2d}/{len(SEARCH_CONFIGS)}: '
              f'h={config["hidden_size"]:3d}, L={config["num_layers"]}, '
              f'lr={config["lr"]:.4f}, seq={seq_len:2d} '
              f'-> val_loss={best_val:.6f} ({t:.1f}s)')

tuning_df = pd.DataFrame(tuning_results)
print(f'\nTotal trials completed: {len(tuning_df)}')

In [ ]:
# Best configuration per model type
best_configs = {}
print('Best Configurations:')
print('=' * 80)

for mt in ['RNN', 'LSTM', 'GRU']:
    subset = tuning_df[tuning_df['model_type'] == mt]
    best_row = subset.loc[subset['best_val_loss'].idxmin()]
    best_configs[mt] = best_row.to_dict()
    print(f'\n{mt}:')
    print(f'  hidden_size={int(best_row["hidden_size"])}, num_layers={int(best_row["num_layers"])}, '
          f'lr={best_row["lr"]}, dropout={best_row["dropout"]}, seq_length={int(best_row["seq_length"])}')
    print(f'  Best val loss: {best_row["best_val_loss"]:.6f}')

In [ ]:
# Tuning visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, mt in zip(axes, ['RNN', 'LSTM', 'GRU']):
    subset = tuning_df[tuning_df['model_type'] == mt]
    pivot = subset.pivot_table(values='best_val_loss', index='hidden_size',
                                columns='num_layers', aggfunc='min')
    sns.heatmap(pivot, annot=True, fmt='.5f', cmap='YlOrRd_r', ax=ax)
    ax.set_title(f'{mt} — Val Loss by Hidden Size & Layers')

plt.suptitle('Hyperparameter Tuning Results', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Final training with best configs
FINAL_EPOCHS = 150
final_models = {}
final_histories = {}
final_times = {}
best_seq_lengths = {}

for mt in ['RNN', 'LSTM', 'GRU']:
    print(f'\nTraining final {mt}...')
    cfg = best_configs[mt]
    seq_len = int(cfg['seq_length'])
    best_seq_lengths[mt] = seq_len

    X_tr_f, y_tr_f = create_sequences(train_scaled, seq_len, TARGET_IDX)
    X_va_f, y_va_f = create_sequences(val_scaled, seq_len, TARGET_IDX)
    loader_f = DataLoader(StockDataset(X_tr_f, y_tr_f), batch_size=BATCH_SIZE, shuffle=True)

    model = SequenceForecaster(
        model_type=mt,
        input_size=N_FEATURES,
        hidden_size=int(cfg['hidden_size']),
        output_size=1,
        num_layers=int(cfg['num_layers']),
        dropout=cfg['dropout']
    )

    history, elapsed = train_model(model, loader_f, X_va_f, y_va_f,
                                    epochs=FINAL_EPOCHS, lr=cfg['lr'], verbose=True)

    final_models[mt] = model
    final_histories[mt] = history
    final_times[mt] = elapsed
    print(f'  {mt} done: {len(history["val_loss"])} epochs in {elapsed:.1f}s')

# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
for mt in ['RNN', 'LSTM', 'GRU']:
    h = final_histories[mt]
    ax1.plot(h['train_loss'], label=mt, color=COLORS[mt], linewidth=1.5)
    ax2.plot(h['val_loss'], label=mt, color=COLORS[mt], linewidth=1.5)

ax1.set_title('Training Loss'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('MSE'); ax1.legend(); ax1.set_yscale('log')
ax2.set_title('Validation Loss'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('MSE'); ax2.legend(); ax2.set_yscale('log')
plt.suptitle('Final Training Curves', fontsize=14)
plt.tight_layout()
plt.show()

---
## 8. THE CRITICAL LESSON: Does the LSTM Actually Work?

This is the **most important section** of this notebook. We will systematically investigate
whether our LSTM (and RNN/GRU) models actually learned to predict stock prices, or whether
they are simply performing a **1-day shifted copy** of the input.

> "If your stock prediction model looks too good to be true, it probably is."


In [ ]:
# 8.1 — Predictions vs Actual overlay (full test set)
# First, get predictions for all models on test set with their best seq_length

all_metrics = {}
all_preds = {}
test_actuals = None

for mt in ['RNN', 'LSTM', 'GRU']:
    seq_len = best_seq_lengths[mt]
    X_te, y_te = create_sequences(test_scaled, seq_len, TARGET_IDX)

    metrics, preds, actuals = evaluate_model(
        final_models[mt], X_te, y_te, scaler, TARGET_IDX, N_FEATURES
    )
    metrics['Training Time'] = f'{final_times[mt]:.1f}s'
    metrics['Parameters'] = final_models[mt].count_parameters()
    all_metrics[mt] = metrics
    all_preds[mt] = preds
    test_actuals = actuals

# Baseline predictions (using default SEQ_LENGTH=30 for baselines)
X_te_base, y_te_base = create_sequences(test_scaled, SEQ_LENGTH, TARGET_IDX)
naive_metrics, naive_preds, naive_actuals = evaluate_baseline(
    naive, X_te_base, y_te_base, scaler, TARGET_IDX, N_FEATURES
)
naive_metrics['Training Time'] = '0s'
naive_metrics['Parameters'] = 0
all_metrics['Naive'] = naive_metrics
all_preds['Naive'] = naive_preds

ma5_metrics, ma5_preds, _ = evaluate_baseline(
    ma5, X_te_base, y_te_base, scaler, TARGET_IDX, N_FEATURES
)
ma5_metrics['Training Time'] = '0s'
ma5_metrics['Parameters'] = 0
all_metrics['MA-5'] = ma5_metrics
all_preds['MA-5'] = ma5_preds

print('All models evaluated on test set.')
print(f'Test set size: {len(test_actuals)} predictions')

In [ ]:
# 8.2 — Full test set prediction overlay
# Use the LSTM's test actuals for consistent x-axis
seq_len_lstm = best_seq_lengths['LSTM']
test_dates = test_df.index[seq_len_lstm:]

fig, ax = plt.subplots(figsize=(18, 7))
ax.plot(test_dates[:len(test_actuals)], test_actuals,
        color='black', linewidth=2, label='Actual', zorder=5)

for mt in ['RNN', 'LSTM', 'GRU']:
    ax.plot(test_dates[:len(all_preds[mt])], all_preds[mt],
            color=COLORS[mt], linewidth=1.2, alpha=0.8, linestyle='--', label=mt)

ax.set_title('Test Set Predictions — At First Glance, Models Look Great!', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Close Price (USD)')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print('\u26a0\ufe0f  The predictions look impressive! But are they really?')
print('Let\'s zoom in to find the truth...')

In [ ]:
# 8.3 — ZOOM INTO 2-WEEK WINDOW to expose the 1-day lag
zoom_start = 50
zoom_end = zoom_start + 14  # 2 weeks = 10 trading days visible

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Top: LSTM only vs actual
ax = axes[0]
ax.plot(range(zoom_end - zoom_start), test_actuals[zoom_start:zoom_end],
        'o-', color='black', linewidth=2, markersize=8, label='Actual', zorder=5)
ax.plot(range(zoom_end - zoom_start), all_preds['LSTM'][zoom_start:zoom_end],
        's--', color=COLORS['LSTM'], linewidth=2, markersize=8, label='LSTM Prediction')

# Draw arrows showing the shift
for i in range(zoom_end - zoom_start - 1):
    ax.annotate('', xy=(i+1, all_preds['LSTM'][zoom_start+i+1]),
                xytext=(i, test_actuals[zoom_start+i]),
                arrowprops=dict(arrowstyle='->', color='red', alpha=0.3, lw=1.5))

ax.set_title('ZOOMED IN: LSTM Prediction vs Actual (2-week window)', fontsize=14)
ax.set_xlabel('Trading Day (relative)')
ax.set_ylabel('Close Price (USD)')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Bottom: Show the shift explicitly
ax2 = axes[1]
ax2.plot(range(zoom_end - zoom_start), test_actuals[zoom_start:zoom_end],
         'o-', color='black', linewidth=2, markersize=8, label='Actual (day t)')
# Shift actual by 1 to create "yesterday's price"
shifted = test_actuals[zoom_start-1:zoom_end-1]
ax2.plot(range(zoom_end - zoom_start), shifted,
         'D--', color='#e74c3c', linewidth=2, markersize=8, label='Yesterday\'s Actual (day t-1)')
ax2.plot(range(zoom_end - zoom_start), all_preds['LSTM'][zoom_start:zoom_end],
         's:', color=COLORS['LSTM'], linewidth=2, markersize=8, label='LSTM Prediction')

ax2.set_title('THE REVEAL: LSTM prediction \u2248 Yesterday\'s Price!', fontsize=14)
ax2.set_xlabel('Trading Day (relative)')
ax2.set_ylabel('Close Price (USD)')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\u274c CRITICAL OBSERVATION:')
print('The LSTM prediction (green) almost perfectly overlaps with yesterday\'s actual price (red)!')
print('The model has learned the TRIVIAL strategy: predict yesterday\'s price.')
print('This is NOT useful prediction — it\'s just a 1-day lag copy.')

In [ ]:
# 8.4 — Naive baseline comparison: show RMSE is similar or BETTER
print('=' * 70)
print('NAIVE BASELINE COMPARISON')
print('=' * 70)

# Use consistent test data for fair comparison
X_te_30, y_te_30 = create_sequences(test_scaled, 30, TARGET_IDX)
naive_m, naive_p, naive_a = evaluate_baseline(naive, X_te_30, y_te_30, scaler, TARGET_IDX, N_FEATURES)

comparison = {'Metric': ['RMSE', 'MAE', 'MAPE']}
for mt in ['RNN', 'LSTM', 'GRU']:
    comparison[mt] = [
        f'{float(all_metrics[mt]["RMSE"]):.4f}',
        f'{float(all_metrics[mt]["MAE"]):.4f}',
        all_metrics[mt]['MAPE']
    ]
comparison['Naive (y_t = y_{t-1})'] = [
    f'{naive_m["RMSE"]:.4f}',
    f'{naive_m["MAE"]:.4f}',
    naive_m['MAPE']
]
comparison['MA-5'] = [
    f'{float(all_metrics["MA-5"]["RMSE"]):.4f}',
    f'{float(all_metrics["MA-5"]["MAE"]):.4f}',
    all_metrics['MA-5']['MAPE']
]

comp_df = pd.DataFrame(comparison).set_index('Metric')
print(comp_df.to_string())

print('\n\u26a0\ufe0f  Look at the numbers carefully:')
print('The Naive baseline (just using yesterday\'s price) is COMPETITIVE with or BETTER than the LSTM!')
print('A model with ZERO parameters matches a model with thousands of parameters.')

In [ ]:
# 8.5 — Directional accuracy: is model better than a coin flip?
def directional_accuracy(actuals, predictions):
    """Percentage of times the model correctly predicts the DIRECTION of price change."""
    actual_direction = np.diff(actuals) > 0  # True = price went up
    pred_direction = (predictions[1:] - actuals[:-1]) > 0  # Predicted direction vs current
    correct = (actual_direction == pred_direction).sum()
    return correct / len(actual_direction) * 100

print('DIRECTIONAL ACCURACY (% of correct up/down predictions)')
print('=' * 60)
print(f'  Random baseline (coin flip): 50.00%')
print()

for mt in ['RNN', 'LSTM', 'GRU']:
    da = directional_accuracy(test_actuals, all_preds[mt])
    better_than_random = 'YES' if da > 50 else 'NO'
    print(f'  {mt:5s}: {da:.2f}%  — Better than coin flip? {better_than_random}')

da_naive = directional_accuracy(naive_a, naive_p)
print(f'  Naive: {da_naive:.2f}%  — (always predicts "no change")')
print()
print('\u274c If directional accuracy is near 50%, the model cannot predict price movements.')
print('This is exactly what the Efficient Market Hypothesis predicts.')

In [ ]:
# 8.6 — Cross-ticker generalization test
print('CROSS-TICKER GENERALIZATION TEST')
print('=' * 60)
print('Training on AAPL, testing on GOOGL, MSFT, TSLA')
print()

generalization_results = []

for ticker in ['GOOGL', 'MSFT', 'TSLA']:
    # Prepare test data for this ticker
    df_other = stock_data[ticker][['Close', 'High', 'Low', 'Open', 'Volume']].copy()
    df_other['Returns'] = df_other['Close'].pct_change()
    df_other['Log_Returns'] = np.log(df_other['Close'] / df_other['Close'].shift(1))
    df_other['Rolling_Mean_5'] = df_other['Close'].rolling(5).mean()
    df_other['Rolling_Mean_20'] = df_other['Close'].rolling(20).mean()
    df_other['Rolling_Std_5'] = df_other['Close'].rolling(5).std()
    df_other = df_other.dropna()

    # Use only the last 15% (test period)
    n_other = len(df_other)
    test_start = int(n_other * 0.85)
    test_other = df_other.iloc[test_start:]

    # Scale with AAPL's scaler (this is a deliberate choice to test generalization)
    test_other_scaled = scaler.transform(test_other.values)

    for mt in ['LSTM']:  # Focus on LSTM for this test
        seq_len = best_seq_lengths[mt]
        X_other, y_other = create_sequences(test_other_scaled, seq_len, TARGET_IDX)

        if len(X_other) == 0:
            continue

        metrics, preds, actuals_other = evaluate_model(
            final_models[mt], X_other, y_other, scaler, TARGET_IDX, N_FEATURES
        )

        # Naive baseline for this ticker
        naive_p_other = inverse_scale_target(
            NaiveBaseline().predict(X_other, TARGET_IDX),
            scaler, TARGET_IDX, N_FEATURES
        )
        naive_rmse = np.sqrt(mean_squared_error(actuals_other, naive_p_other))

        da = directional_accuracy(actuals_other, preds)

        generalization_results.append({
            'Ticker': ticker,
            'LSTM RMSE': f'{metrics["RMSE"]:.2f}',
            'Naive RMSE': f'{naive_rmse:.2f}',
            'LSTM Directional Acc': f'{da:.1f}%',
            'LSTM beats Naive?': 'YES' if metrics['RMSE'] < naive_rmse else 'NO'
        })

        print(f'  {ticker}: LSTM RMSE={metrics["RMSE"]:.2f}, Naive RMSE={naive_rmse:.2f}, '
              f'Dir Acc={da:.1f}%, Beats Naive: {"YES" if metrics["RMSE"] < naive_rmse else "NO"}')

gen_df = pd.DataFrame(generalization_results)
gen_df

In [ ]:
# 8.7 — Returns prediction instead of price prediction
print('RETURNS PREDICTION (much harder and more honest)')
print('=' * 60)

# Create a returns-based dataset
returns_data = aapl['Close'].pct_change().dropna().values.astype(np.float32).reshape(-1, 1)

# Split
n_ret = len(returns_data)
ret_train_end = int(n_ret * 0.70)
ret_val_end = int(n_ret * 0.85)

ret_scaler = MinMaxScaler(feature_range=(-1, 1))
ret_train = ret_scaler.fit_transform(returns_data[:ret_train_end])
ret_val = ret_scaler.transform(returns_data[ret_train_end:ret_val_end])
ret_test = ret_scaler.transform(returns_data[ret_val_end:])

# Create sequences (univariate: just returns)
ret_seq_len = 30
X_ret_train, y_ret_train = create_sequences(ret_train, ret_seq_len, target_idx=0)
X_ret_val, y_ret_val = create_sequences(ret_val, ret_seq_len, target_idx=0)
X_ret_test, y_ret_test = create_sequences(ret_test, ret_seq_len, target_idx=0)

print(f'Returns training data: X={X_ret_train.shape}, y={y_ret_train.shape}')
print(f'Returns test data:     X={X_ret_test.shape}, y={y_ret_test.shape}')

# Train LSTM on returns
ret_loader = DataLoader(StockDataset(X_ret_train, y_ret_train), batch_size=BATCH_SIZE, shuffle=True)
ret_model = SequenceForecaster('LSTM', input_size=1, hidden_size=64, num_layers=1)
ret_history, _ = train_model(ret_model, ret_loader, X_ret_val, y_ret_val,
                              epochs=100, lr=0.001, verbose=False)

# Evaluate
ret_model.eval()
with torch.no_grad():
    ret_preds_scaled = ret_model(X_ret_test.to(DEVICE)).cpu().numpy()

ret_preds = ret_scaler.inverse_transform(ret_preds_scaled).flatten()
ret_actuals = ret_scaler.inverse_transform(y_ret_test.numpy()).flatten()

# Metrics
ret_rmse = np.sqrt(mean_squared_error(ret_actuals, ret_preds))
ret_corr = np.corrcoef(ret_actuals, ret_preds)[0, 1]

# Directional accuracy on returns
ret_dir_acc = (np.sign(ret_preds) == np.sign(ret_actuals)).mean() * 100

print(f'\nReturns prediction RMSE: {ret_rmse:.6f}')
print(f'Returns prediction correlation: {ret_corr:.4f}')
print(f'Returns directional accuracy: {ret_dir_acc:.1f}%')
print(f'Random baseline: 50.0%')
print()
if ret_dir_acc < 52:
    print('\u274c LSTM cannot predict return direction better than random!')
else:
    print('Result: marginally above random, but not practically useful.')

In [ ]:
# 8.8 — Returns scatter plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter: predicted vs actual returns
axes[0].scatter(ret_actuals, ret_preds, alpha=0.3, s=10, color='#3498db')
axes[0].plot([ret_actuals.min(), ret_actuals.max()],
             [ret_actuals.min(), ret_actuals.max()],
             'r--', linewidth=1, label='Perfect prediction')
axes[0].axhline(0, color='gray', linewidth=0.5)
axes[0].axvline(0, color='gray', linewidth=0.5)
axes[0].set_title('LSTM: Predicted vs Actual Returns', fontsize=13)
axes[0].set_xlabel('Actual Return')
axes[0].set_ylabel('Predicted Return')
axes[0].legend()

# Distribution of predictions vs actuals
axes[1].hist(ret_actuals, bins=80, alpha=0.5, color='black', label='Actual Returns', density=True)
axes[1].hist(ret_preds, bins=80, alpha=0.5, color=COLORS['LSTM'], label='Predicted Returns', density=True)
axes[1].set_title('Distribution: Actual vs Predicted Returns', fontsize=13)
axes[1].set_xlabel('Return')
axes[1].legend()

plt.suptitle('Returns Prediction — The Honest Test', fontsize=14)
plt.tight_layout()
plt.show()

print('Notice: The model predicts returns clustered near ZERO (the mean).')
print('It cannot capture the spread/variance of actual returns.')
print('This is equivalent to predicting "no change" — the naive strategy.')

In [ ]:
# 8.9 — Comprehensive final metrics table WITH naive baseline

print('\n' + '=' * 90)
print('FINAL COMPREHENSIVE COMPARISON TABLE')
print('=' * 90)

final_table = {}
for mt in ['RNN', 'LSTM', 'GRU', 'Naive', 'MA-5']:
    final_table[mt] = all_metrics[mt].copy()

# Add directional accuracy
for mt in ['RNN', 'LSTM', 'GRU']:
    da = directional_accuracy(test_actuals, all_preds[mt])
    final_table[mt]['Dir. Accuracy'] = f'{da:.1f}%'

final_table['Naive']['Dir. Accuracy'] = '50.0%'
final_table['MA-5']['Dir. Accuracy'] = 'N/A'

ftable_df = pd.DataFrame(final_table).T
print(ftable_df.to_string())

print('\n--- VERDICT ---')
print('The naive baseline (0 parameters, 0 training time) performs COMPARABLY to the LSTM.')
print('Thousands of parameters and minutes of training buy you essentially NOTHING')
print('for next-day stock price prediction.')

In [ ]:
# 8.10 — Visual summary: the uncomfortable truth
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: RMSE comparison
models_list = ['RNN', 'LSTM', 'GRU', 'Naive', 'MA-5']
rmse_vals = [float(all_metrics[m]['RMSE']) for m in models_list]
bar_colors = [COLORS.get(m, '#95a5a6') for m in models_list]
bars = axes[0].bar(models_list, rmse_vals, color=bar_colors, edgecolor='white')
for bar, val in zip(bars, rmse_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{val:.2f}', ha='center', fontweight='bold', fontsize=9)
axes[0].set_title('RMSE (lower is better)', fontsize=13)
axes[0].set_ylabel('RMSE (USD)')

# Plot 2: Parameters vs RMSE (cost of complexity)
for mt in ['RNN', 'LSTM', 'GRU']:
    axes[1].scatter(all_metrics[mt]['Parameters'], float(all_metrics[mt]['RMSE']),
                    color=COLORS[mt], s=200, label=mt, zorder=5, edgecolors='black')
axes[1].axhline(float(all_metrics['Naive']['RMSE']), color='gray', linestyle='--',
                label=f'Naive RMSE={float(all_metrics["Naive"]["RMSE"]):.2f}')
axes[1].set_title('Complexity vs Performance', fontsize=13)
axes[1].set_xlabel('Parameters')
axes[1].set_ylabel('RMSE (USD)')
axes[1].legend(fontsize=9)

# Plot 3: Training time vs RMSE
for mt in ['RNN', 'LSTM', 'GRU']:
    axes[2].scatter(final_times[mt], float(all_metrics[mt]['RMSE']),
                    color=COLORS[mt], s=200, label=mt, zorder=5, edgecolors='black')
axes[2].axhline(float(all_metrics['Naive']['RMSE']), color='gray', linestyle='--',
                label=f'Naive RMSE')
axes[2].set_title('Training Time vs Performance', fontsize=13)
axes[2].set_xlabel('Training Time (seconds)')
axes[2].set_ylabel('RMSE (USD)')
axes[2].legend(fontsize=9)

plt.suptitle('The Uncomfortable Truth: Complexity Buys Nothing for Stock Prediction', fontsize=14)
plt.tight_layout()
plt.show()

---
## 9. Analysis: Why Stock Prediction Fails

### The 1-Day Shift Illusion
When you overlay LSTM predictions on actual prices, they look remarkably accurate. But this
is an **illusion**. The model has learned the simplest possible strategy: predict that
tomorrow's price will be the same as today's price. Since stock prices are highly
autocorrelated at the price level (but NOT at the returns level), this strategy looks
"accurate" but provides zero useful information.

### Why This Happens

1. **MSE loss incentivizes conservatism**: MSE penalizes large errors quadratically.
   The safest strategy under MSE is to predict the conditional mean, which for a
   random walk is approximately yesterday's price.

2. **Stock prices are approximately a random walk**: The EMH implies that price changes
   are unpredictable. The best predictor of P(t+1) is P(t) + drift.

3. **No information content in history**: Unlike weather (physical laws) or airline
   passengers (seasonal patterns), stock prices efficiently incorporate all available
   information. The historical price sequence contains no additional predictive signal.


In [ ]:
# 9.1 — Visual proof of the 1-day shift
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Show correlation between LSTM prediction(t) and actual(t-1)
ax = axes[0]
lstm_preds = all_preds['LSTM']
actual_shifted = test_actuals[:-1]     # actual(t-1)
lstm_preds_aligned = lstm_preds[1:]    # prediction(t)

ax.scatter(actual_shifted, lstm_preds_aligned, alpha=0.3, s=10, color=COLORS['LSTM'])
ax.plot([actual_shifted.min(), actual_shifted.max()],
        [actual_shifted.min(), actual_shifted.max()],
        'r--', linewidth=2, label='Perfect y=x line')
corr_shift = np.corrcoef(actual_shifted, lstm_preds_aligned)[0, 1]
ax.set_title(f'LSTM prediction(t) vs Actual(t-1) — Correlation: {corr_shift:.4f}', fontsize=14)
ax.set_xlabel('Actual Price at t-1 (yesterday)')
ax.set_ylabel('LSTM Prediction for t (today)')
ax.legend(fontsize=11)

# Show correlation between LSTM prediction(t) and actual(t)
ax2 = axes[1]
ax2.scatter(test_actuals, lstm_preds, alpha=0.3, s=10, color=COLORS['LSTM'])
ax2.plot([test_actuals.min(), test_actuals.max()],
         [test_actuals.min(), test_actuals.max()],
         'r--', linewidth=2, label='Perfect y=x line')
corr_same = np.corrcoef(test_actuals, lstm_preds)[0, 1]
ax2.set_title(f'LSTM prediction(t) vs Actual(t) — Correlation: {corr_same:.4f}', fontsize=14)
ax2.set_xlabel('Actual Price at t')
ax2.set_ylabel('LSTM Prediction for t')
ax2.legend(fontsize=11)

plt.tight_layout()
plt.show()

print(f'Correlation of LSTM prediction with YESTERDAY\'s price: {corr_shift:.4f}')
print(f'Correlation of LSTM prediction with TODAY\'s price:     {corr_same:.4f}')
print()
if corr_shift > corr_same:
    print('\u274c LSTM prediction correlates MORE with yesterday\'s price than today\'s!')
    print('This proves the model is just copying yesterday\'s price with a 1-day lag.')
else:
    print('Both correlations are very high — because prices are highly autocorrelated.')
    print('The model tracks the slow trend but cannot predict day-to-day movements.')

### Common Pitfalls in Stock Prediction with Deep Learning

**1. Look-Ahead Bias (Data Leakage)**
- Using future information in features (e.g., same-day High/Low to predict Close)
- Not splitting chronologically (random splits leak future patterns into training)
- Using indicators calculated with future data points

**2. The "Impressive Plot" Trap**
- Overlaying predictions on prices always looks good because of high autocorrelation
- The correct evaluation is on **returns** or **directional accuracy**, not price level
- Any model predicting "no change" will have low RMSE on prices

**3. Survivorship Bias**
- Only testing on stocks that still exist (successful companies)
- Companies that went bankrupt are excluded from historical datasets

**4. Overfitting to Specific Market Regimes**
- A model trained on a bull market will fail in a bear market
- Financial markets are non-stationary: statistical properties change over time

**5. Transaction Costs**
- Even if a model had slight predictive power, transaction costs (commissions, spread, slippage)
  would likely eliminate any theoretical profit


In [ ]:
# 9.2 — Price prediction error vs returns prediction error
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Price prediction errors
price_errors = test_actuals - all_preds['LSTM']
axes[0].hist(price_errors, bins=50, color=COLORS['LSTM'], alpha=0.7, edgecolor='white', density=True)
axes[0].axvline(0, color='red', linewidth=1)
axes[0].set_title(f'Price Prediction Errors (LSTM)\nMean={np.mean(price_errors):.2f}, Std={np.std(price_errors):.2f}', fontsize=12)
axes[0].set_xlabel('Error (USD)')

# Returns prediction errors
ret_errors = ret_actuals - ret_preds
axes[1].hist(ret_errors, bins=50, color='#e74c3c', alpha=0.7, edgecolor='white', density=True)
axes[1].axvline(0, color='red', linewidth=1)
axes[1].set_title(f'Returns Prediction Errors (LSTM)\nMean={np.mean(ret_errors):.6f}, Std={np.std(ret_errors):.6f}', fontsize=12)
axes[1].set_xlabel('Error (return units)')

plt.suptitle('Error Distribution: Prices vs Returns', fontsize=14)
plt.tight_layout()
plt.show()

print('Price errors look small in absolute terms but contain no useful signal.')
print('Return errors are essentially the same as the return distribution itself.')
print('The model has not learned to reduce uncertainty about future returns.')

In [ ]:
# 9.3 — Information content analysis: does the model add value beyond naive?
print('INFORMATION CONTENT ANALYSIS')
print('=' * 60)

# Theil's U statistic: compares model to naive
def theils_u(actual, predicted, naive_pred):
    """Theil's U statistic. U < 1 means model beats naive. U = 1 means equal. U > 1 means worse."""
    model_mse = mean_squared_error(actual, predicted)
    naive_mse = mean_squared_error(actual, naive_pred)
    return np.sqrt(model_mse / naive_mse) if naive_mse > 0 else float('inf')

# Ensure same length for comparison
min_len = min(len(test_actuals), len(naive_a))
for mt in ['RNN', 'LSTM', 'GRU']:
    preds_mt = all_preds[mt][:min_len]
    act_mt = test_actuals[:min_len]
    naive_mt = naive_a[:min_len]

    u = theils_u(act_mt, preds_mt, naive_mt)
    print(f'  {mt:5s} Theil\'s U = {u:.4f}  — {"BEATS naive" if u < 1 else "DOES NOT beat naive"}')

print()
print('Theil\'s U interpretation:')
print('  U < 1.0: Model provides information beyond naive baseline')
print('  U = 1.0: Model is equivalent to naive baseline')
print('  U > 1.0: Model is WORSE than naive baseline')
print()
print('For stock prices, U is typically very close to 1.0,')
print('confirming the LSTM adds no real predictive value.')

---
## 10. Conclusion: What We Learned

### Summary of Findings

| Aspect | Result |
|--------|--------|
| **LSTM vs Naive Baseline** | LSTM does NOT meaningfully beat the naive baseline (predict yesterday's price) |
| **Directional Accuracy** | Near 50% — no better than a coin flip |
| **The "Shift" Illusion** | LSTM predictions are a lagged copy of actual prices |
| **Returns Prediction** | Model predicts near-zero returns (the mean) — no useful signal |
| **Cross-Ticker Generalization** | No meaningful transfer from AAPL to other stocks |
| **Theil's U** | Close to 1.0 — model adds no information beyond naive |

### The Key Lesson

> **LSTMs (and RNNs, GRUs) are powerful tools for sequence modeling, but they cannot extract
> predictive signal from data that contains no signal.** Stock prices, under the Efficient
> Market Hypothesis, are approximately random walks. No amount of model complexity can
> predict a random walk better than the naive baseline.

### When DO Recurrent Models Work for Time Series?

LSTMs excel when the data has:
1. **Strong temporal patterns**: Weather (physical laws), electricity load (daily/weekly cycles)
2. **Seasonal structure**: Air passengers, retail sales
3. **Physical constraints**: Sensor data, manufacturing processes
4. **Learnable dynamics**: Heart rate, speech signals

LSTMs **struggle** when:
1. **Data is a random walk**: Stock prices, exchange rates
2. **Information is efficiently priced**: Financial markets
3. **The signal-to-noise ratio is very low**: High-frequency trading data

### Ethical Considerations

This notebook is a reminder that **impressive-looking results can be misleading**.
In the real world, people lose money by trusting ML models for stock prediction.
Always:
- Compare against simple baselines
- Look at directional accuracy, not just RMSE
- Zoom in on predictions to check for lagging
- Test on out-of-sample data from different time periods
- Consider whether the underlying process is actually predictable

### Progression in This Series
- **Notebook 01** (Air Passengers): LSTM works well — clear seasonal patterns
- **Notebook 02** (Jena Climate): LSTM works well — physical laws govern weather
- **Notebook 03** (Stock Prices): LSTM FAILS — market efficiency prevents prediction
- This progression teaches you to think critically about when deep learning is appropriate.


In [ ]:
# Final visualization: the narrative arc
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Case 1: Air Passengers (LSTM works)
axes[0].text(0.5, 0.7, 'Case Study 1\nAir Passengers', fontsize=14,
             ha='center', va='center', transform=axes[0].transAxes, fontweight='bold')
axes[0].text(0.5, 0.4, 'LSTM WORKS', fontsize=20,
             ha='center', va='center', transform=axes[0].transAxes,
             color=COLORS['LSTM'], fontweight='bold')
axes[0].text(0.5, 0.2, 'Strong seasonal pattern\nClear trend', fontsize=10,
             ha='center', va='center', transform=axes[0].transAxes, color='gray')
axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1)
axes[0].set_frame_on(True)
for spine in axes[0].spines.values():
    spine.set_color(COLORS['LSTM'])
    spine.set_linewidth(3)
axes[0].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

# Case 2: Climate (LSTM works)
axes[1].text(0.5, 0.7, 'Case Study 2\nJena Climate', fontsize=14,
             ha='center', va='center', transform=axes[1].transAxes, fontweight='bold')
axes[1].text(0.5, 0.4, 'LSTM WORKS', fontsize=20,
             ha='center', va='center', transform=axes[1].transAxes,
             color=COLORS['LSTM'], fontweight='bold')
axes[1].text(0.5, 0.2, 'Physical laws\nMultivariate patterns', fontsize=10,
             ha='center', va='center', transform=axes[1].transAxes, color='gray')
axes[1].set_xlim(0, 1); axes[1].set_ylim(0, 1)
for spine in axes[1].spines.values():
    spine.set_color(COLORS['LSTM'])
    spine.set_linewidth(3)
axes[1].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

# Case 3: Stock Prices (LSTM fails)
axes[2].text(0.5, 0.7, 'Case Study 3\nStock Prices', fontsize=14,
             ha='center', va='center', transform=axes[2].transAxes, fontweight='bold')
axes[2].text(0.5, 0.4, 'LSTM FAILS', fontsize=20,
             ha='center', va='center', transform=axes[2].transAxes,
             color=COLORS['RNN'], fontweight='bold')
axes[2].text(0.5, 0.2, 'Efficient Market Hypothesis\nRandom walk behavior', fontsize=10,
             ha='center', va='center', transform=axes[2].transAxes, color='gray')
axes[2].set_xlim(0, 1); axes[2].set_ylim(0, 1)
for spine in axes[2].spines.values():
    spine.set_color(COLORS['RNN'])
    spine.set_linewidth(3)
axes[2].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

plt.suptitle('The Narrative Arc: When Does Deep Learning Work for Time Series?', fontsize=16)
plt.tight_layout()
plt.show()

print('Notebook 03 complete.')
print('Key takeaway: Always question whether your problem is actually predictable')
print('before throwing a deep learning model at it.')